# 🔗 SGLang 前缀共享策略 — 多场景深度剖析

**本文目标**：通过具体场景深入理解 RadixAttention 在不同负载下的表现。

读完这篇你会理解：
- 工具调用场景的前缀共享模式
- 多模态场景的 visual token 共享
- 批量推理的公共前缀复用
- 共享 vs 隔离的权衡

## 1. 场景一: Agent 多轮工具调用

### 1.1 前缀共享模式

```
5 个 Agent 请求, 相同 system prompt + 不同用户问题

请求 A: [Sys] [User: "北京天气"] [TC: get_weather] [Result: 35C] [Asst: "35度"]
请求 B: [Sys] [User: "上海天气"] [TC: get_weather] [Result: 32C] [Asst: "32度"]
请求 C: [Sys] [User: "搜索 AI"]   [TC: search]      [Result: ...]  [Asst: "关于AI..."]
请求 D: [Sys] [User: "北京天气"] [TC: get_weather] [Result: 35C] [Asst: "北京35度"]
请求 E: [Sys] [User: "计算 1+1"]  [TC: calculate]   [Result: 2]    [Asst: "结果是2"]

Radix Tree 结构:
                    ┌──────────┐
                    │  Root     │
                    └────┬─────┘
                         │
              ┌──────────┴──────────┐
              │  [System Prompt]     │  ← 5/5 请求共享 (100% 命中)
              │  2,000 tokens        │
              └──────────┬──────────┘
                         │
        ┌────────────────┼────────────────┬──────────────┐
        │                │                │              │
   ┌────┴────┐     ┌─────┴─────┐   ┌──────┴──────┐  ┌───┴───┐
   │"北京天气"│     │"上海天气"  │   │"搜索 AI"    │  │"计算  │
   │A,D 共享!│     │           │   │             │  │ 1+1"  │
   └────┬────┘     └─────┬─────┘   └──────┬──────┘  └───┬───┘
        │                │                │              │
   ┌────┴────┐          ...              ...            ...
   │TC: get_ │
   │weather  │  ← 工具调用格式可能共享 (如果 constrained)
   │Result:  │
   │35C      │  ← 完全相同! A 和 D 的 result 一样 → 共享!
   └────┬────┘
        │
   ┌────┴────┐
   │"35度"    │  ← A
   │"北京35度" │  ← D (不同 → 分叉)
   └─────────┘

关键发现:
  1. System Prompt 100% 共享 (最大收益)
  2. 相同用户问题的请求: 前缀也共享 (如 A 和 D)
  3. Tool call 格式: constrained generation 使得格式一致 → 更多共享
  4. Tool result: 如果相同则共享 (cache 命中), 但一般不相同
```

### 1.2 P99 延迟改善

```
无前缀缓存:
  请求 D (与 A 的前 2K+user+TC 相同) → 重新 prefill 所有 tokens
  → TTFT = prefill(2K+20+30) ≈ 200ms

有 RadixAttention:
  请求 D → 前缀命中 → 只 prefill 新 tokens (result + assistant)
  → TTFT ≈ prefill(1000+50) ≈ 100ms → 50% 改善

对于 100 QPS 的服务:
  → P99 TTFT 显著下降
  → 批量处理 (相同 system prompt) 时改善最明显
```

## 2. 场景二: 多模态 Visual Token 共享

### 2.1 同一张图的共享

```
场景: 10 个用户上传了同一张 viral 图片, 问不同的问题

请求 A: [同一张图 576 tokens] [文本: "这是什么?"]
请求 B: [同一张图 576 tokens] [文本: "颜色是什么?"]
请求 C: [另一张图 576 tokens] [文本: "描述一下"]

Radix Tree:
  ┌──────────┐
  │  Root     │
  └────┬─────┘
       │
  ┌────┴──────────────┐
  │ [Visual Token 0]  │
  │ [Visual Token 1]  │
  │ ...               │  ← 576 tokens
  │ [Visual Token 575]│
  └────┬──────────────┘
       │
  ┌────┴────┐      ┌──────────┐
  │ "这是什么?"│      │ [另一张图] │  ← 请求 C: 独立分支
  │ A        │      │ 576 tokens │
  ├──────────┤      └─────┬─────┘
  │ "颜色是?" │            │
  │ B        │       "描述一下"
  └──────────┘

收益:
  请求 A 和 B 共享 576 visual tokens 的 KV Cache
  → 每个请求节省 576 tokens 的 KV Cache (~288 KB → 不大)
  → 但节省 prefill 时间: 视觉 prefill 通常 ~300-500ms
  → 10 个请求共享 → 节省 ~3s CPU/GPU 时间

vLLM APC 能做到吗?
  如果 block_size=16, 576 tokens = 36 blocks
  图片像素完全相同 → block hash 相同 → 可以命中
  但如果图片被 resize/压缩 → visual tokens 不同 → 无法命中
  
  SGLang: token-level → 即使图片被轻微处理, 只要大部分 tokens 相同就共享
```

### 2.2 多图场景

```
请求: 5 张图 + 文本问题
  [Img1 576][Img2 576][Img3 576][Img4 576][Img5 576][Text 50]
  
如果多个请求的图片子集相同:
  A: [Img1][Img2][Img3][Img4][Img5][TextA]
  B: [Img1][Img2][Img3][Img6][Img7][TextB]
  
  Radix Tree:
    → Img1, Img2, Img3 共享 (前 1728 tokens)
    → Img4/Img5 vs Img6/Img7 分叉

  在 SGLang 中这是自动的, 不需要人工指定哪些图相同
```

## 3. 场景三: 批量推理 (Batch Processing)

```
任务: 对 1000 篇文档做同样的结构化提取

传统做法 (vLLM):
  每个文档独立请求, system prompt 通过 APC 共享
  但 extraction instruction 在每个请求中相同
  → 受限于 block 边界对齐

SGLang 做法:
  [System Prompt] [Extraction Instruction] → 100% 共享
  [Document 1] [Output 1]  ─┐
  [Document 2] [Output 2]  ─┤ 分叉
  ...                       │
  [Document 1000] [Output 1000] ─┘
  
  收益:
  - System prompt + instruction = ~500 tokens → 999 个请求共享
  - 如果有 constrained generation → 输出的 JSON 结构也可能部分共享
  
  量化:
  无共享: 1000 × 500 = 500,000 tokens KV Cache = 250 MB
  有共享: 500 tokens KV Cache = 0.25 MB → 节省 99.9%
  时间节省: 999 × prefill(500 tokens) ≈ 999 × 50ms = 50s → 几乎为 0
```

## 4. 共享 vs 隔离的权衡

### 4.1 什么时候不应该共享？

```
场景: 多租户 SaaS, 不同客户的 system prompt 不同

Client A: "You are a bank assistant. Be formal."
Client B: "You are a casual chatbot. Be friendly."

→ 它们的 system prompt 完全不同 → 不应共享
→ Radix Tree 会自动分叉, 反而增加了树的复杂度

解决方案:
  SGLang 的 session-based isolation:
  为不同的 "prefix domain" 创建独立的 radix tree
  → client_id 作为隔离键

  但这需要额外配置, 不是默认行为。
```

### 4.2 Radix Tree 的维护开销

```
Radix Tree 的大小:
  每个 token = 1 个节点
  10K 请求 × 平均 3K tokens = 30M 节点
  → 每个节点 ~40 bytes metadata
  → ~1.2 GB CPU 内存 (仅 metadata!)

缓解措施:
  1. LRU 淘汰: ref_count=0 的节点自动回收
  2. 子树剪枝: 整个子树无引用 → 一起回收
  3. 时间限制: 超过 TTL 的节点强制回收
  4. 大小限制: max_tree_size → 超过则淘汰最老的
```